In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

from xgboost import XGBRegressor as XGBR

## XGB

In [ ]:
data_path = r"path/Train_Validation_Verification_Data_revised.xlsx"

# Train sheet = training set
train_data = pd.read_excel(data_path, sheet_name="Train")

# Val sheet = validation set
val_data = pd.read_excel(data_path, sheet_name="Val")


feature_train = train_data.iloc[:, :10].to_numpy(dtype=np.float32)
label_train = train_data.iloc[:, -1].to_numpy(dtype=np.float32)

feature_val = val_data.iloc[:, :10].to_numpy(dtype=np.float32)
label_val = val_data.iloc[:, -1].to_numpy(dtype=np.float32)

print("Train:", feature_train.shape, label_train.shape)
print("Val   :", feature_val.shape, label_val.shape)

In [ ]:
xgb = XGBR(n_estimators=375, max_depth=3, random_state=300).fit(
    feature_train,
    label_train
)

xgb.score(feature_val, label_val)

## MQR

In [ ]:
data_path = r"path/Train_Validation_Verification_Data_revised.xlsx"

# Train sheet = training set
train_data = pd.read_excel(data_path, sheet_name="Train")

# Val sheet = validation set
val_data = pd.read_excel(data_path, sheet_name="Val")


single_bc_path = data_path = r"path/Train_Validation_Verification_Data_revised.xlsx"
single_bc_data = pd.read_excel(single_bc_path, sheet_name="1BC")

print("Single BC data shape:", single_bc_data.shape)
print("Single BC columns:")
print(single_bc_data.columns.tolist())

display(single_bc_data.head())


bc_cols = [
    "Cellulose",
    "Hemicellulose",
    "Lignin",
    "PE",
    "PS",
    "PP",
    "PET",
    "PVC",
    "Starch"
]

time_col = "Time"
label_col = "TG"

bc_name_map = {
    "Cellulose": "Cellulose",
    "Hemicellulose": "Hemicellulose",
    "Lignin": "Lignin",
    "PE": "PE",
    "PS": "PS",
    "PP": "PP",
    "PET": "PET",
    "PVC": "PVC",
    "Starch": "Starch"
}


def build_single_bc_tg_matrix(single_bc_data, bc_cols, bc_name_map):
    single_tg_dict = {}

    for bc in bc_cols:
        single_col = bc_name_map[bc]
        one_bc_curve = single_bc_data.loc[
            single_bc_data[single_col] == 1,
            ["Time", "TG(wt.%)"]
        ].copy()

        one_bc_curve = one_bc_curve.reset_index(drop=True)

        single_tg_dict[bc] = one_bc_curve["TG(wt.%)"].to_numpy(dtype=np.float32)

    first_bc = bc_cols[0]
    first_single_col = bc_name_map[first_bc]

    time_array = single_bc_data.loc[
        single_bc_data[first_single_col] == 1,
        "Time"
    ].reset_index(drop=True).to_numpy(dtype=np.float32)

    single_tg_df = pd.DataFrame(single_tg_dict)
    single_tg_df.insert(0, time_col, time_array)

    return single_tg_df


single_tg_df = build_single_bc_tg_matrix(
    single_bc_data=single_bc_data,
    bc_cols=bc_cols,
    bc_name_map=bc_name_map
)


def build_mqr_features_old_fusion(
    df,
    single_tg_df,
    bc_cols,
    time_col,
    points_per_experiment=574,
    use_percent_composition=True
):
    df = df.copy().reset_index(drop=True)

    bcs_values = df[bc_cols].to_numpy(dtype=np.float32)

    max_bcs = np.nanmax(bcs_values)

    if use_percent_composition and max_bcs <= 1.5:
        bcs_values = bcs_values * 100.0

    data_time = df[time_col].to_numpy(dtype=np.float32)
    single_time = single_tg_df[time_col].to_numpy(dtype=np.float32)

    time_indices = np.array([
        np.argmin(np.abs(single_time - t))
        for t in data_time
    ])

    max_time_diff = np.max(np.abs(data_time - single_time[time_indices]))

    single_tg_values = single_tg_df[bc_cols].to_numpy(dtype=np.float32)
    single_tg_for_each_row = single_tg_values[time_indices, :]

    # Features mixture
    fused_features = bcs_values * single_tg_for_each_row / 100.0

    fused_df = pd.DataFrame(
        fused_features,
        columns=bc_cols,
        index=df.index
    )

    return fused_df


X_train_df = build_mqr_features_old_fusion(
    df=train_data,
    single_tg_df=single_tg_df,
    bc_cols=bc_cols,
    time_col=time_col,
    points_per_experiment=574,
    use_percent_composition=True
)

X_val_df = build_mqr_features_old_fusion(
    df=val_data,
    single_tg_df=single_tg_df,
    bc_cols=bc_cols,
    time_col=time_col,
    points_per_experiment=574,
    use_percent_composition=True
)

y_train = train_data[label_col].to_numpy(dtype=np.float32).ravel()
y_val = val_data[label_col].to_numpy(dtype=np.float32).ravel()

X_train = X_train_df.copy()
X_val = X_val_df.copy()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

display(X_train_df.head())
display(X_val_df.head())

In [5]:
poly = PolynomialFeatures(
    degree=2,
    interaction_only=True,
    include_bias=False
)

X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)

feature_names = poly.get_feature_names_out(bc_cols)

model = LinearRegression()
model.fit(X_train_poly, y_train)

feature_names = poly.get_feature_names_out(bc_cols)

In [6]:
y_pred = model.predict(X_train_poly)
r2_train = r2_score(y_train, y_pred)
mean_squared_error(y_train, y_pred, squared=False)

7.849749

In [7]:
r2_train

0.9535985379469741

In [8]:
y_pred = model.predict(X_val_poly)
r2_val = r2_score(y_val, y_pred)
mean_squared_error(y_val, y_pred, squared=False)

6.9684877

In [9]:
r2_val

0.9585815044626734